# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a template for loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa/) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)

metadata = dataset.metadata
print(f"Name: {metadata.name}\nDescription: {metadata.description}")
print(f"Published: {getattr(metadata, 'datePublished', 'Unknown')}")
print(f"License: {getattr(metadata, 'license', 'Not specified')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Entities in the dataset are referenced by their `@id` fields, following the Croissant schema.

In [ ]:
# List all record sets by `@id` and their field `@id`s
record_sets = dataset.metadata.record_sets
for rs in record_sets:
    print(f"Record Set @id: {rs.id} - {getattr(rs, 'name', '')}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    Field @id: {field.id} - {getattr(field, 'name', '')}, dataType: {getattr(field, 'data_type', '')}")
    print()

## 3. Data Extraction
Load data from selected record sets into DataFrames for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Collect all record set @ids
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

# Load a sample from each record set
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for record set {record_set_id}")

# For further demonstration, choose the first record set
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id:
    print(f"\nColumns for records in '{main_record_set_id}':\n{dataframes[main_record_set_id].columns.tolist()}")
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We demonstrate with a numeric field from the main record set.

In [ ]:
import numpy as np

# List numeric fields of the main record set
if main_record_set_id:
    main_rs = next((rs for rs in dataset.metadata.record_sets if rs.id == main_record_set_id), None)
    numeric_fields = [f for f in main_rs.fields if getattr(f, 'data_type', None) in ['Float', 'Integer', 'Number', 'schema:Float', 'schema:Integer', 'schema:Number']]
    if numeric_fields:
        numeric_field_id = numeric_fields[0].id
        numeric_field_name = getattr(numeric_fields[0], 'name', numeric_field_id)

        # Drop missing values for numeric analysis
        df = dataframes[main_record_set_id].copy()
        # Coerce values to numeric, as mlcroissant output may be str
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        
        threshold = df[numeric_field_id].mean() if not np.isnan(df[numeric_field_id].mean()) else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_name} (@id={numeric_field_id}) > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_name} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # If there is a categorical field, group by it
        categorical_fields = [f for f in main_rs.fields if getattr(f, 'data_type', None) in ['Text', 'String', 'schema:Text', 'schema:String']]
        if categorical_fields:
            group_field_id = categorical_fields[0].id
            print(f"\nGrouped mean by {getattr(categorical_fields[0], 'name', group_field_id)} (@id={group_field_id}):")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            display(grouped_df.head())
        else:
            print("No categorical/text field found for grouping.")
    else:
        print("No numeric fields found in main record set.")
else:
    print("No record sets available.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and numeric_fields:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_name} ({numeric_field_id})")
    plt.xlabel(numeric_field_name)
    plt.ylabel("Count")
    plt.show()
    
    if categorical_fields:
        plt.figure(figsize=(10, 5))
        top_categories = df[group_field_id].value_counts().index[:5]
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df[df[group_field_id].isin(top_categories)])
        plt.title(f"{numeric_field_name} by {getattr(categorical_fields[0], 'name', group_field_id)}")
        plt.xlabel(getattr(categorical_fields[0], 'name', group_field_id))
        plt.show()
else:
    print("Insufficient numeric or categorical fields for visualization.")

## 6. Conclusion
In this notebook, we've demonstrated how to load, overview, and explore a mass spectrometry protein dataset using the `mlcroissant` library. We reviewed the Croissant schema entities by `@id`, loaded records, performed basic filtering and transformation, and visualized distributions. Further biological or scientific insights can be derived using the dataset's rich annotation and experimental context.
